# Session 4: Scraping with Beautiful Soup Home work

We will scrape **https://www.moviessda.com/tamilrockers/**  and turn one webpage into a Pandas dataframe.

By the end, you should be able to:

- request a webpage
- check whether the request worked
- parse HTML
- identify a repeated container
- extract titles, prices, ratings, availability and links
- create and inspect a dataframe
- clean and save the scraped data

> Workflow: **Request → Parse → Select → Extract → Store → Validate**

## Before scraping

Understand the site's html structure.

## 1. Install packages if needed

### This cell installs the three external libraries used in the notebook:

- `requests` downloads the webpage.
- `beautifulsoup4` reads and searches the HTML.
- `pandas` turns the extracted information into a table.

In [21]:
pip install requests beautifulsoup4 pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: C:\Users\Sherlyn Petricia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 2. Import libraries

### We are loading four tools:

- `requests` will fetch the webpage.
- `BeautifulSoup` will turn raw HTML into a searchable structure.
- `pandas`, shortened to `pd`, will create and analyse our dataframe.
- `urljoin` will turn incomplete links from the webpage into full URLs.

Running an import makes these tools available to the notebook.

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin

#### import requests
A Python library that sends requests to websites and downloads their content. Without it, Python has no way to ask a website for its contents.

#### from urllib.parse import urljoin
A function that combines a website's base URL with an incomplete (relative) link to create a full webpage address. We use it because many websites store links as relative paths instead of complete URLs.


Website
   │
   ▼
requests
(download page)
   │
   ▼
Beautiful Soup
(read HTML)
   │
   ▼
urljoin
(fix incomplete links)
   │
   ▼
Pandas
(store everything as a dataframe)

## 3. Store the URL

### We are saving the webpage address inside a variable called `url`.

The URL is text, so it must be placed inside quotation marks. Storing it in a variable means we can reuse the address later without typing it repeatedly.

The second line displays the value so we can confirm that the variable contains the expected address.

In [20]:
url = "https://www.moviessda.com/tamilrockers/"
url

'https://www.moviessda.com/tamilrockers/'

## 4. Request the webpage

This is the first time Python communicates with a website.

When we write:

```python
response = requests.get(url)
```

the following happens:

1. Python sends an HTTP **GET request** to the website.
2. The website's server receives that request.
3. The server sends back a **response**.
4. We store that response in a variable called `response`.

Think of it like ordering food:

- **You** → make the request.
- **Restaurant** → prepares the order.
- **Delivery bag** → the response.

The response contains much more than the webpage itself. It also contains:
- the status code
- headers
- cookies
- and the HTML that generated the page.

We haven't downloaded a spreadsheet—we've downloaded the webpage itself.


### What we are about to do

This cell sends an HTTP request to the website.

- `requests.get(url, timeout=30)` asks the server for the page stored in `url`.
- `timeout=30` prevents Python from waiting forever if the website does not respond.
- The server's reply is saved in a variable called `response`.
- Writing `response` on the final line displays a short summary, usually something such as `<Response [200]>`.

At this stage, `response` contains the status code, headers and webpage HTML.

In [13]:
response = requests.get(url, timeout=30)
response

<Response [200]>

### Optional: Check the response's status code.

A status code is the server's short message about what happened:

- `200` means the request succeeded.
- `403` means access was refused.
- `404` means the page was not found.
- `500` means the server encountered an error.

We should check this before trying to parse the page.

In [7]:
response.status_code

200

## 5. Look at the raw HTML

The browser turns HTML into something beautiful using CSS. But, Python doesn't see colours, buttons or layouts.
It only receives the underlying HTML.

The next two lines do two different jobs:

```python
html = response.text
```
stores the webpage's HTML inside a variable called `html`.

```python
print(html[:1000])
```

prints only the **first 1,000 characters**.

Why only the first thousand?

A webpage can contain tens of thousands of characters. Printing everything is a lot.

`[:1000]` is called **slicing** and simply means:

> "Show me the first thousand characters."

### Separate the webpage's HTML from the rest of the response.

- `response.text` contains the HTML as one long string of text.
- We store that string in a variable called `html`.
- `html[:1000]` uses slicing to keep only the first 1,000 characters.
- `print()` displays that shortened sample.

We inspect only the beginning because printing the full page would overwhelm the notebook.

In [14]:
html = response.text
print(html[:1000])

<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>Tamilrockers | Tamilrockers 2026 Tamil Movies Download</title>
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<meta name="description" content="Tamilrockers 2026 Movies Download Tamilrockers 2026 Tamil Movies Download Tamilrockers Full Movies Download Tamilrockers.com">
<meta name="keywords" content="tamilrockers 2026 movies download tamilrockers 2026 tamil movies download tamilrockers full movies download tamilrockers.com">
<link rel="canonical" href="https://www.moviessda.com/tamilrockers/">
<link rel="shortcut icon" href="/img/favicon.ico">
<link rel="stylesheet" type="text/css" href="/style.css?v=1" media="all">
<meta name="theme-color" content="#efbacc">
</head>
<body>
<div class="line">
  <font size="4px">Moviesda.Mobi</font>
  <br/>
  <small>Name Of Quality</small>
</div>
<div class="bottom" align="center">Like our <a href="https://t.me/moviesda-2025-page"><font color="red">Facebook Fan

In [9]:
# html = response.text
# print(html[10000:15000])

Look for repeated structures such as:

```html
<article class="product_pod">
<p class="price_color">£51.77</p>
```

## 6. Parse the HTML

Right now `html` is just one very long string of text. Beautiful Soup converts that text into something we can search.

Think of it like this:

Before Beautiful Soup:
```
One giant wall of text
```

After Beautiful Soup:
```
A searchable tree of elements
```

Instead of searching through thousands of characters ourselves, we can now ask questions like:
- Find the first heading.
- Find every book.
- Find every price.
- Find every link.

### Turning the raw HTML string into a Beautiful Soup object.

- `html` is currently plain text.
- `"html.parser"` tells Beautiful Soup which parser to use.
- The parsed page is stored in `soup`.
- `type(soup)` checks what kind of Python object was created.

After this step, we can search the webpage by tags, classes and other HTML features.

In [15]:
soup = BeautifulSoup(html, "html.parser")
type(soup)

bs4.BeautifulSoup

## 7. Find the page heading

### What we are about to do

We are asking Beautiful Soup to find the first `<h1>` heading on the page.

- `soup.find("h1")` searches the parsed HTML.
- The matching HTML element is stored in `heading`.
- Displaying `heading` shows both the tag and its contents.

This is our first simple test that Beautiful Soup can locate an element successfully.

In [16]:
heading = soup.find_all("h1")
heading

[]

In [17]:
heading_new = soup.find("h1")
heading_new

### The `heading` variable currently contains an HTML element such as `<h1>All products</h1>`.

`get_text(strip=True)` removes the surrounding tags and returns only the visible words. `strip=True` also removes extra spaces and line breaks from the beginning and end.

In [24]:
heading_new = soup.select_one("div.line")

In [25]:
heading_new.get_text(strip=True)

'Moviesda.MobiName Of Quality'

In [26]:
print(heading_new.get_text(strip=True))

Moviesda.MobiName Of Quality


## 8. Find one Movie

Every book lives inside:

```html
<article class="div.f">
```

Every book has the **same HTML structure**. That repetition is exactly what makes scraping possible.
Rather than writing code twenty times, we teach Python how to recognise one book card. Then Python repeats the same process for every card.

### Locating the first repeated book card.

`article.product_pod` is a CSS selector:

- `article` refers to the HTML tag.
- The period means “class”.
- `product_pod` is the class name shared by every book card.
- `select_one()` returns only the first matching card.

We save that complete card in `first_book` so we can practise extracting one field at a time before looping over all books.

In [28]:
first_movie = soup.select_one("div.f")
print(first_movie)

<div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=11">Tamilrockers (2026) Movies</a> </div>


In [29]:
first_movie

<div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=11">Tamilrockers (2026) Movies</a> </div>

`article.product_pod` means:

> Find an `article` element whose class is `product_pod`.


In [30]:
all_movies = soup.select("div.f")

In [31]:
all_movies

[<div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=11">Tamilrockers (2026) Movies</a> </div>,
 <div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=11">Tamilrockers (2025) Movies</a> </div>,
 <div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=1">Tamilrockers (2024) Movies</a> </div>,
 <div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=2">Tamilrockers (2023) Movies</a> </div>,
 <div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=3">Tamilrockers (2022) Movies</a> </div>,
 <div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=4">Tamilrockers (2021) Movies</a> </div>,
 <div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=5">Tamilrockers (2020) Movies</a> </div>,
 <div class="f"> <img alt="[+]" src="/img/dir.gif"/> <a href="https://gotopage.top/?ref=

In [32]:
# for movies in all_movies:
#     print(movies)

## 9. Extract one title

### Inside the first movie card, locate the link that contains the title.

The selector `h3 a` means:

> Find an `<a>` link located inside an `<h3>` heading.

We store the matching HTML element in `title_element` and display it so we can inspect its visible text and attributes.

In [33]:
movie_name = first_movie.select_one("h3 a")
movie_name

In [34]:
movie_name = first_movie.select_one("h3")
movie_name

### The full movie title is stored in the link's `title` attribute.

- `title_element["title"]` extracts that attribute's value.
- We save it in a variable called `title`.
- The final line displays the extracted title.

This is different from `get_text()`: here we are reading an HTML attribute rather than the visible words between the tags.

In [35]:
# title = movie_name["title"]
# title

- `.get_text()` extracts visible text.
- `["title"]` extracts the value of an HTML attribute.


## 10. Extract the Movie Link Address

### Extracting the Movie link from the first movie card.



In [37]:
link_url = first_movie.select_one("a")["href"]

## 11. Extract movie format

### We are extracting the availability message from the first book card.



In [38]:
movie_format = first_book.select_one("a").get_text(strip=True)

print("Movie Format:", movie_format)

Movie Format: Tamilrockers (2026) Movies


## 12. Extract the release year

The movie icon image path (src) is stored as an attribute inside the <img> tag rather than visible text.

This cell:

Finds the image element inside the card.

Stores the element in icon_element.

Uses .get("src") to retrieve its image file path attribute.

We expect a result such as /img/dir.gif.

In [39]:
import re


title_element = first_book.select_one("a")


text = title_element.get_text(strip=True) if title_element else ""
match = re.search(r"\d{4}", text)
release_year = match.group() if match else None

print("Release Year:", release_year)

Release Year: 2026


## 13. Extract the link

What we are about to do

The webpage stores a relative link rather than a complete web address inside the <a> tag. title_element.get("href") retrieves that incomplete path.
urljoin(url, relative_link) combines the site's base URL with the relative path to create a complete, usable movie URL.

In [40]:
from urllib.parse import urljoin


relative_link = first_book.select_one("a").get("href")


full_movie_url = urljoin(url, relative_link)

print("Full Movie URL:", full_movie_url)

Full Movie URL: https://gotopage.top/?ref=11


In [42]:
relative_link = first_movie.select_one("a").get("href")


full_movie_url = urljoin(url, relative_link)

print("Full Movie URL:", full_movie_url)

Full Movie URL: https://gotopage.top/?ref=11


In [43]:
relative_link

'https://gotopage.top/?ref=11'

## 14. Store one movie as a dictionary

### Grouping the first movie's extracted fields into a dictionary.

A dictionary stores information as `key: value` pairs:

- the keys will later become dataframe column names
- the values will become the cells in one row

This dictionary represents one complete observation: one book.

In [45]:
import re
from urllib.parse import urljoin


first_movie = soup.select_one("div.f")


title = first_movie.select_one("a").get_text(strip=True)
movie_format = first_movie.select_one("a").get_text(strip=True)


year_match = re.search(r"\d{4}", title)
release_year = year_match.group() if year_match else None


relative_link = first_movie.select_one("a").get("href")
full_movie_url = urljoin(url, relative_link)

first_movie_data = {
    "title": title,
    "release_year": release_year,
    "movie_format": movie_format,
    "movie_url": full_movie_url
}

first_movie_data

{'title': 'Tamilrockers (2026) Movies',
 'release_year': '2026',
 'movie_format': 'Tamilrockers (2026) Movies',
 'movie_url': 'https://gotopage.top/?ref=11'}

Dictionary is very close to one dataframe row:

- dictionary = row
- key = column
- value = cell

## 15. Find all movie cards

### Now find **every** movie card on the page.

select() returns all matching elements, unlike select_one(), which returns only the first.

The resulting collection is stored in movies.

len(movies) counts how many category cards were found.

The webpage shows multiple movie categories (typically around 10 to 15 per page), so checking this length serves as an important validation check.

In [46]:
movies = soup.select("div.f")

print("Total movies found:", len(movies))

Total movies found: 9


## 16. Loop through the movies

### What we are about to do: understand the loop

`movies contains all the separate movie-card elements found on the page. Rather than copying the same title-extraction code for each card, we use a for loop.

Read the first line as:

For each individual movie inside the collection called movies, repeat the indented instructions.

For every card, Python:

gives the current card the temporary name movie

searches inside that card for the a tag

extracts the visible link text using .get_text(strip=True)

saves it temporarily as title

prints the title

moves to the next card and repeats

The indentation matters: both indented lines belong to the loop. When the loop finishes, all movie titles on the page will have been printed.

In [47]:
for movie in movies:
    title = movie.select_one("a").get_text(strip=True)
    print(title)

Tamilrockers (2026) Movies
Tamilrockers (2025) Movies
Tamilrockers (2024) Movies
Tamilrockers (2023) Movies
Tamilrockers (2022) Movies
Tamilrockers (2021) Movies
Tamilrockers (2020) Movies
Tamilrockers Movies Collection
Tamilrockers HD Mobile Movies


## 17. Build a list of rows

What we are about to do

So far we've extracted information from one movie.

Now we want all the movies on the page.

Create an empty list called rows.

Visit each movie card one at a time.

Extract the title, release year, format, and link.

Store those four pieces of information as a dictionary.

Add that dictionary to our growing list.

When the loop finishes, rows will contain one dictionary for every movie card.

Later, Pandas will convert that list directly into a dataframe.

### Build one row per movie
This is the main scraping loop.

First, rows = [] creates an empty list that will collect our results.

Then, for each movie card, Python:

finds the movie link

extracts the title

extracts the release year using regex

extracts the movie format

builds the full movie URL

creates one dictionary containing those fields

appends that dictionary to rows

After the loop, rows should contain all extracted dictionaries—one for every movie card. len(rows) checks that count.

In [48]:
import re
from urllib.parse import urljoin

# Create an empty list.
# We'll store one dictionary per movie in this list.
rows = []

# Loop through every movie card we found on the webpage.
# On the first iteration, 'movie' is the first movie card.
# On the second iteration, it's the second movie card, and so on.
for movie in movies:

    # Find the <a> tag inside the movie card.
    # This contains both the title text and the relative link.
    title_element = movie.select_one("a")
    
    # Extract visible title text safely
    title = title_element.get_text(strip=True) if title_element else ""

    # Extract 4-digit release year using regex search
    year_match = re.search(r"\d{4}", title)
    release_year = year_match.group() if year_match else None

    # Add one dictionary to our list.
    # Each dictionary becomes one row in our final dataframe.
    rows.append({

        # The movie title is stored directly as the visible text
        # inside the <a> tag.
        "title": title,

        # The release year is extracted from the title text
        # using regex matching (r"\d{4}").
        "release_year": release_year,

        # The movie format/category string matches the full title text.
        "movie_format": title,

        # The webpage stores a relative link such as:
        # tamilrockers-2026-movies.html
        # urljoin() combines it with the website's base URL
        # to create a complete, usable link.
        "movie_url": urljoin(
            url,
            title_element.get("href") if title_element else ""
        )
    })

# Count how many dictionaries (movies) we collected.
len(rows)

9

### Inspecting the first dictionary stored in the `rows` list.

Python uses zero-based indexing, so `rows[0]` means “show the first item”. This lets us confirm that the loop stored the expected fields before we create a dataframe.

In [49]:
rows[0]

{'title': 'Tamilrockers (2026) Movies',
 'release_year': '2026',
 'movie_format': 'Tamilrockers (2026) Movies',
 'movie_url': 'https://gotopage.top/?ref=11'}

## 18. Create a dataframe

This is the moment where web scraping meets Pandas.

Currently:

```
rows
```

is a **list of dictionaries**.

Pandas knows how to turn that structure into a table automatically.

Think about the mapping:

- one dictionary → one row
- dictionary keys → column names
- dictionary values → cells

After this line, everything you've already learned in Pandas works exactly the same.


### Convert the list of dictionaries into a Pandas dataframe.

Pandas interprets the structure automatically:

- each dictionary becomes one row
- each dictionary key becomes a column
- each dictionary value becomes a cell

`df.head()` then displays the first five scraped books so we can inspect the result.

In [50]:
df = pd.DataFrame(rows)
df.head()

,title,release_year,movie_format,movie_url
0,Tamilrockers (2026) Movies,2026,Tamilrockers (2026) Movies,https://gotopage.top/?ref=11
1,Tamilrockers (2025) Movies,2025,Tamilrockers (2025) Movies,https://gotopage.top/?ref=11
2,Tamilrockers (2024) Movies,2024,Tamilrockers (2024) Movies,https://gotopage.top/?ref=1
3,Tamilrockers (2023) Movies,2023,Tamilrockers (2023) Movies,https://gotopage.top/?ref=2
4,Tamilrockers (2022) Movies,2022,Tamilrockers (2022) Movies,https://gotopage.top/?ref=3


### `df.shape` reports the dataframe's dimensions as:

```text
(number of rows, number of columns)
```

We expect 20 rows because the homepage contains 20 book cards. This is another check that our scraper found the expected number of observations.

In [51]:
df.shape

(9, 4)

In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   title         9 non-null      str  
 1   release_year  7 non-null      str  
 2   movie_format  9 non-null      str  
 3   movie_url     9 non-null      str  
dtypes: str(4)
memory usage: 420.0 bytes


## 19. Clean the release year

Clean the release year

The scraped release year is extracted as text, so Pandas currently treats it as an object/string.
This cell creates a clean numeric column:

select release_year

convert the text values to whole numbers using .astype("Int64") (or float)

save the result back to release_year

We convert missing values safely so the transformation remains transparent and ready for analysis.

In [53]:
# Convert the extracted string year into a clean integer column
df["release_year"] = df["release_year"].astype("Int64")

# View title and release year columns
df[["title", "release_year"]].head()

,title,release_year
0,Tamilrockers (2026) Movies,2026
1,Tamilrockers (2025) Movies,2025
2,Tamilrockers (2024) Movies,2024
3,Tamilrockers (2023) Movies,2023
4,Tamilrockers (2022) Movies,2022


This:

selects the release year text

converts the result to a whole number (Int64)

saves it as a clean numeric column


## 20. Convert release status to numbers

Categorize release status

The ratings are words such as One, Two, and Five. On our movie site, release years represent how recent a movie is. We want a numeric status code (or label) that makes movies easier to sort, filter, and summarize by release status.

First, we create a function or dictionary mapping release years to status codes (e.g., 3 for New Release, 2 for Recent, 1 for Older). Then, we map every movie year to write the corresponding status into a new column called release_status.

In [54]:
# Create a dictionary mapping release years to numeric status levels
# 3 = Current (2026), 2 = Recent (2024-2025), 1 = Older (2023 & earlier)
status_map = {
    2026: 3,
    2025: 2,
    2024: 2,
    2023: 1,
    2022: 1
}

# Map the release_year column to create release_status
# .fillna(1) gives any earlier/unmapped years a default status of 1
df["release_status"] = df["release_year"].map(status_map).fillna(1).astype(int)

# View title, release_year, and release_status columns
df[["title", "release_year", "release_status"]].head()

,title,release_year,release_status
0,Tamilrockers (2026) Movies,2026,3
1,Tamilrockers (2025) Movies,2025,2
2,Tamilrockers (2024) Movies,2024,2
3,Tamilrockers (2023) Movies,2023,1
4,Tamilrockers (2022) Movies,2022,1


In [55]:
df.head(5)

,title,release_year,movie_format,movie_url,release_status
0,Tamilrockers (2026) Movies,2026,Tamilrockers (2026) Movies,https://gotopage.top/?ref=11,3
1,Tamilrockers (2025) Movies,2025,Tamilrockers (2025) Movies,https://gotopage.top/?ref=11,2
2,Tamilrockers (2024) Movies,2024,Tamilrockers (2024) Movies,https://gotopage.top/?ref=1,2
3,Tamilrockers (2023) Movies,2023,Tamilrockers (2023) Movies,https://gotopage.top/?ref=2,1
4,Tamilrockers (2022) Movies,2022,Tamilrockers (2022) Movies,https://gotopage.top/?ref=3,1


## 21. Analyse the scraped data

In [56]:
df.sort_values("release_year", ascending=False)[[
    "title",
    "release_year",
    "release_status"
]].head(10)

,title,release_year,release_status
0,Tamilrockers (2026) Movies,2026,3
1,Tamilrockers (2025) Movies,2025,2
2,Tamilrockers (2024) Movies,2024,2
3,Tamilrockers (2023) Movies,2023,1
4,Tamilrockers (2022) Movies,2022,1
5,Tamilrockers (2021) Movies,2021,1
6,Tamilrockers (2020) Movies,2020,1
7,Tamilrockers Movies Collection,<NA>,1
8,Tamilrockers HD Mobile Movies,<NA>,1


In [58]:
df["release_year"].mean().round(2)

np.float64(2023.0)

Count how many movies belong to each release status.

value_counts() counts the frequency of every release status value.

sort_index() arranges the result in status order from 1 to 3 (older to newer) rather than by frequency.

This shows the distribution of movie release recency on the page.

In [60]:
df["release_status"].value_counts().sort_index()

release_status
1    6
2    2
3    1
Name: count, dtype: int64

### Sorting movie

sorts all movies by the numeric release year column

puts the newest movies first because ascending=False

selects only the title, release year and release status columns

displays the first ten rows

This answers a simple reporting-style question: which movies on this page are the newest releases?

In [61]:
df[df["release_status"] == 3][[
    "title",
    "release_year"
]].sort_values("release_year", ascending=False)

,title,release_year
0,Tamilrockers (2026) Movies,2026


## 22. Validate the scrape

In [62]:
print("Rows:", len(df))

print("\nMissing values:")
print(df.isna().sum())

print("\nRelease status range:")
print(df["release_status"].min(), "to", df["release_status"].max())

print("\nRelease year range:")
print(df["release_year"].min(), "to", df["release_year"].max())

Rows: 9

Missing values:
title             0
release_year      2
movie_format      0
movie_url         0
release_status    0
dtype: int64

Release status range:
1 to 3

Release year range:
2020 to 2026


A scraper can run without an error and still collect the wrong data. Compare a few rows manually with the webpage.


## 23. Select final columns

### Create a cleaner final dataframe containing only the columns we want to keep.

`.copy()` makes an independent copy of those selected columns. This is useful because later changes to `books_df` will not accidentally modify the original `df`.

In [63]:
movies_df = df[[
    "title",
    "release_year",
    "release_status",
    "movie_format",
    "movie_url"
]].copy()

movies_df.head()

,title,release_year,release_status,movie_format,movie_url
0,Tamilrockers (2026) Movies,2026,3,Tamilrockers (2026) Movies,https://gotopage.top/?ref=11
1,Tamilrockers (2025) Movies,2025,2,Tamilrockers (2025) Movies,https://gotopage.top/?ref=11
2,Tamilrockers (2024) Movies,2024,2,Tamilrockers (2024) Movies,https://gotopage.top/?ref=1
3,Tamilrockers (2023) Movies,2023,1,Tamilrockers (2023) Movies,https://gotopage.top/?ref=2
4,Tamilrockers (2022) Movies,2022,1,Tamilrockers (2022) Movies,https://gotopage.top/?ref=3
